# Bashkir (bak) — Full NLP Pipeline with Custom Stanza Models

Bashkir (Kipchak branch) is written in Cyrillic. TurkicNLP provides Beta-quality Apertium FST morphological analysis, custom-trained Stanza neural models for POS tagging, lemmatisation, and dependency parsing, and NLLB-200 embeddings and translation.

In [ ]:
# Install TurkicNLP
# pip install turkicnlp          # core (tokenization, transliteration)
# pip install "turkicnlp[stanza]"  # adds POS, lemma, depparse, NER
# pip install "turkicnlp[nllb]"    # adds cross-lingual embeddings + translation
# pip install "turkicnlp[all]"     # all optional dependencies

In [ ]:
import turkicnlp
from turkicnlp import Pipeline

## 1. Download Models

## 2. Script Detection and Cyrillic ↔ Latin Transliteration

Bashkir uses Cyrillic script. The transliteration system allows conversion to Latin script for processing or analysis.

In [ ]:
from turkicnlp.scripts import Script
from turkicnlp.scripts.detector import detect_script
from turkicnlp.scripts.transliterator import Transliterator

# Bashkir Cyrillic text
cyrl = "Мин мәктәпкә барам."
print("Script Detection:")
print(f"  Detected: {detect_script(cyrl).name}")
print()

# Cyrillic -> Turkic Common Alphabet (Latin)
try:
    t = Transliterator("bak", source=Script.CYRILLIC, target=Script.COMMON_TURKIC)
    common = t.transliterate(cyrl)
    print(f"Cyrillic:        {cyrl}")
    print(f"Turkic Common:   {common}")
    
    # Reverse: Turkic Common -> Cyrillic
    t_back = Transliterator("bak", source=Script.COMMON_TURKIC, target=Script.CYRILLIC)
    cyrl_restored = t_back.transliterate(common)
    print(f"Restored:        {cyrl_restored}")
    print(f"Round-trip match: {cyrl == cyrl_restored}")
except Exception as e:
    print(f"⚠ Note: Transliteration to Turkic Common Alphabet (Latin) may not be fully supported for Bashkir: {e}")
    print("  For Cyrillic-based languages, use Script.LATIN as alternative")

In [ ]:
turkicnlp.download('bak')

## 2. Tokenisation

In [ ]:
nlp_tok = Pipeline("bak", processors=["tokenize"])
doc = nlp_tok("Мин мәктәпкә барам.")
for sent in doc.sentences:
    print([w.text for w in sent.words])

## 3. Morphological Analysis (Apertium FST — Beta)

In [ ]:
nlp = Pipeline(
    "bak",
    processors=["tokenize", "morph"],
    morph_backend="apertium",
)
doc = nlp("Мин мәктәпкә барам.")
for w in doc.words:
    print(f"{w.text:<18} lemma={w.lemma:<12} feats={w.feats}")

## 4. POS Tagging, Lemmatisation, and Dependency Parsing (Custom Stanza)

TurkicNLP includes custom-trained Stanza models for Bashkir, providing POS tagging, lemmatisation, and dependency parsing.

In [ ]:
nlp_parse = Pipeline(
    "bak",
    processors=["tokenize", "pos", "lemma", "depparse"],
)

doc = nlp_parse("Бер журнал был айҙың һанында уның тормошон микроскоп аҫтына ала.")
print(f"{'Word':<20} {'UPOS':<8} {'Lemma':<20} {'Head':<5} {'Deprel'}")
print("-" * 60)
for w in doc.words:
    print(f"{w.text:<20} {w.upos:<8} {w.lemma:<20} {w.head!s:<5} {w.deprel}")

## 5. Full Pipeline with CoNLL-U Export

In [ ]:
nlp_full = Pipeline(
    "bak",
    processors=["tokenize", "morph", "pos", "lemma", "depparse"],
    morph_backend="apertium",
)
doc = nlp_full("Башҡортостан — Рәсәй Федерацияһы субъекты.")
print(doc.to_conllu())

## 6. Translation

In [ ]:
turkicnlp.download("bak", processors=["translate"])
trans = Pipeline("bak", processors=["translate"], translate_tgt_lang="rus_Cyrl")
doc = trans("Башҡортостан — Рәсәй Федерацияһы субъекты.")
print("RU:", doc.translation)